# Probing space saving, the compression reading of the band tasks

One figure, drawn from a table the collection already writes. Nothing here re-runs a model.

`collect_band_tasks` scores the seven hierarchical band-classification tasks of
`probe_lib.BAND_TASKS`, after Pagani et al., at every probe point of a geometry, and stores for
each cell the prequential codelength and three readings of it:

* `SV` $= 1 - L(D)/L_{\text{uniform}}(D)$, the space saving of the trained model;
* `SV_shuffled`, the same probe on permuted labels, which is the memorisation control: a value at
  or below zero says the probe is not fitting noise;
* `SV_random_init`, an untrained architecture of identical shape, which separates what the model
  learned from what its architecture affords.

The figure below plots all three against the probe stage. It is built from the columns this
repository actually computes; the layout follows the description in Deliverable 1 rather than
reproducing the source paper's figure exactly, so check it against the paper before citing it as
a reproduction.


In [1]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

# ---- where the collection wrote its tables -------------------------------------------------
def find_repo():
    here = Path.cwd().resolve()
    for base in [here] + list(here.parents):
        if (base / "chronos" / "bayesian" / "probe_lib.py").exists():
            return base
    raise FileNotFoundError("run this from inside the patchAliasing checkout")

REPO  = find_repo()
BAYES = REPO / "chronos" / "bayesian"

# The analysis notebook writes to <root>/full/data, where <root> is the checkout on a local run
# and the Drive folder on Colab. Look in every place it could be, newest match first.
DRIVE = Path("/content/drive/MyDrive/patchAliasing")
if not DRIVE.exists():
    import importlib.util
    if importlib.util.find_spec("google.colab") is not None:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE = Path("/content/drive/MyDrive/patchAliasing")

ROOTS = [DRIVE, BAYES, BAYES / "_run"]
CAND = []
for r in ROOTS:
    if r.exists():
        CAND += sorted(r.rglob("*mdl_bandtasks*.parquet"))
CAND = sorted(set(CAND), key=lambda q: q.stat().st_mtime, reverse=True)
if not CAND:
    raise FileNotFoundError(
        "no band-task table found. Looked under: " + ", ".join(str(r) for r in ROOTS) +
        ". Run the collection first, or set CAND to the file holding 02_mdl_bandtasks.parquet")
print("reading:", *[str(q) for q in CAND], sep="\n  ")

OUT = CAND[0].parent.parent / "probing"          # beside the run the table came from
(OUT / "figures").mkdir(parents=True, exist_ok=True)
BT = pd.concat([pd.read_parquet(p) for p in CAND], ignore_index=True)
print(f"{len(CAND)} file(s), {len(BT)} rows, {BT['model'].nunique()} geometries")
display(BT.head())

FileNotFoundError: run this from inside the patchAliasing checkout

In [ ]:
# ---- configuration --------------------------------------------------------------------------
GEOMETRY   = "p16-s16"      # the panel drawn in detail; the published tokeniser geometry
TASK_ORDER = ["Mid", "L", "H", "LL", "LH", "HL", "HH"]     # by level, as in tab:bandTasks
C_TRAIN, C_SHUF, C_RAND = "#1f4e79", "#8c8c8c", "#c81e3c"


def stage_order(df):
    """Encoder blocks, then decoder blocks, then the two output points, in that order."""
    s = list(dict.fromkeys(df["stage"]))
    def key(x):
        if x.startswith("enc_"): return (0, int(x.split("_")[1]))
        if x.startswith("dec_"): return (1, int(x.split("_")[1]))
        return (2, ["output_reg", "output_head"].index(x) if x in
                   ("output_reg", "output_head") else 9)
    return sorted(s, key=key)


def nice(stage):
    if stage.startswith("enc_"): return f"enc {int(stage.split('_')[1]) + 1}"
    if stage.startswith("dec_"): return f"dec {int(stage.split('_')[1]) + 1}"
    return {"output_reg": "pre-proj", "output_head": "head (h4)"}.get(stage, stage)


def report(p):
    import time
    st = p.stat()
    print(f"    wrote {p.name}  {st.st_size / 1024:.0f} kB  "
          f"{time.strftime('%H:%M:%S', time.localtime(st.st_mtime))}  -> {p.parent}")

## Figure, space saving across the probe points

Left, one line per band task for the reference geometry, with the two controls drawn beneath them:
where the trained curve sits well above both, the stage carries information about the frequency
band that neither label noise nor the untrained architecture supplies. Right, the same quantity at
the output head for every geometry, which is the comparison across the design.


In [ ]:
def spacesaving_page(geometry=GEOMETRY, fname=None):
    d = BT[BT["model"] == geometry]
    if d.empty:
        raise ValueError(f"{geometry} not in the table; have {sorted(BT['model'].unique())}")
    stages = stage_order(d)
    xs = np.arange(len(stages))
    tasks = [t for t in TASK_ORDER if t in set(d["task"])] or sorted(d["task"].unique())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.2, 5.2),
                                   gridspec_kw=dict(width_ratios=[1.15, 1.0], wspace=0.22))

    # left, one line per task, plus the two controls averaged over tasks
    for i, t in enumerate(tasks):
        sub = d[d["task"] == t].set_index("stage").reindex(stages)
        ax1.plot(xs, sub["SV"], color=C_TRAIN, lw=1.4, marker="o", ms=3.4, mew=0,
                 alpha=0.35 + 0.65 * (i + 1) / len(tasks))
        ax1.annotate(t, (xs[-1], sub["SV"].iloc[-1]), textcoords="offset points",
                     xytext=(4, 0), fontsize=7, color=C_TRAIN, va="center")
    for col, colour, lab in (("SV_shuffled", C_SHUF, "shuffled labels"),
                             ("SV_random_init", C_RAND, "random init")):
        if col in d and d[col].notna().any():
            m = d.groupby("stage")[col].mean().reindex(stages)
            ax1.plot(xs, m, color=colour, lw=1.3, ls="--", marker="s", ms=3.0, mew=0)
    ax1.axhline(0, color="0.75", lw=0.8, zorder=0)
    ax1.set_xticks(xs); ax1.set_xticklabels([nice(s) for s in stages], rotation=45,
                                            ha="right", fontsize=7.5)
    ax1.set_ylabel("space saving  $SV = 1 - L(D)/L_{unif}(D)$", fontsize=9)
    ax1.set_title(f"{geometry}: compression of the band tasks, by probe point", fontsize=10.5)
    ax1.tick_params(labelsize=8)
    ax1.legend(handles=[Line2D([0], [0], color=C_TRAIN, lw=1.6, marker="o", ms=4, mew=0,
                               label="trained model, one line per band task"),
                        Line2D([0], [0], color=C_SHUF, lw=1.4, ls="--", marker="s", ms=3.4,
                               mew=0, label="shuffled labels (memorisation control)"),
                        Line2D([0], [0], color=C_RAND, lw=1.4, ls="--", marker="s", ms=3.4,
                               mew=0, label="random initialisation (architecture control)")],
               fontsize=7.5, loc="lower left", framealpha=0.9)

    # right, the output head across the design
    head = "output_head" if (BT["stage"] == "output_head").any() else stage_order(BT)[-1]
    h = BT[BT["stage"] == head]
    order = sorted(h["model"].unique(), key=lambda m: (int(m.split("-")[0][1:]),
                                                       int(m.split("-s")[1])))
    for t in tasks:
        sub = h[h["task"] == t].set_index("model").reindex(order)
        ax2.plot(np.arange(len(order)), sub["SV"], lw=1.3, marker="o", ms=3.4, mew=0, label=t)
    ax2.axhline(0, color="0.75", lw=0.8, zorder=0)
    ax2.set_xticks(np.arange(len(order))); ax2.set_xticklabels(order, rotation=60, ha="right",
                                                               fontsize=7)
    ax2.set_title(f"at the {nice(head)} probe point, every geometry", fontsize=10.5)
    ax2.tick_params(labelsize=8); ax2.legend(fontsize=7, ncol=4, title="band task",
                                             title_fontsize=7)

    fig.suptitle("Probing space saving on the seven hierarchical band tasks", fontsize=12.5,
                 y=0.995)
    fig.text(0.5, 0.945,
             "SV = 1 - L(D)/L_uniform(D) from the prequential codelength; higher is more "
             "compressible, so more of the band label is linearly present in that state. "
             "Zero is the uniform code.",
             ha="center", fontsize=8.5, color="0.3")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    p = OUT / "figures" / (fname or f"FIG_SV_{geometry}.png")
    fig.savefig(p, dpi=200, bbox_inches="tight")
    fig.savefig(p.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig); report(p); report(p.with_suffix(".pdf"))
    return p


page = spacesaving_page()

## The numbers behind it

The same content as a table, so a value quoted in the report can be traced to a row.


In [ ]:
piv = (BT.pivot_table(index=["model", "stage"], columns="task", values="SV")
         .reindex(columns=[t for t in TASK_ORDER if t in set(BT["task"])]))
piv["mean"] = piv.mean(axis=1)
display(piv.round(3))
piv.to_csv(OUT / "space_saving.csv")
print("wrote", OUT / "space_saving.csv")